In [ ]:
from IPython.display import Markdown, display

import gcsfs
import pandas as pd

from update_vars import PROCESSED_GCS, DIGEST_DICT, abbrev_month

In [ ]:
#analysis_name = "Alameda-Contra Costa Transit District"

In [ ]:
operator_summary_url = f"{PROCESSED_GCS}{DIGEST_DICT.operator_summary}_{abbrev_month}.parquet"

operator_df = pd.read_parquet(
    operator_summary_url,
    filesystem = gcsfs.GCSFileSystem(),
    filters=[
        ("Day Type", "==", "Weekday"),
        #("Analysis Name", "==", analysis_name)
    ],
).drop_duplicates(
    subset = ["Analysis Name", "Date"]
).reset_index(drop=True)

bridge contains schedule_gtfs_dataset_name to analysis_name to ntd_id
* fanout can occur if multiple feeds are combined to same analysis name
   * Redding Remix, Schedule, Flex are all Redding name
   * they share the same ntd_id
* how should query be set up to remove the deduping needed?
   * can just download dim_annual_agency_information with the most recent
   * make sure the join here works to attach NTD profile information
   * crosswalk would keep analysis_name - ntd_id and join with dim_annual_agency_deduped_and_downloaded

In [ ]:
#ntd_url = f"{PROCESSED_GCS}{DIGEST_DICT.ntd_profile}_{abbrev_month}.parquet"
ntd_columns = ["analysis_name", "county_name", "caltrans_district_full", 
               "service_area_sq_miles", "service_area_pop", "primary_uza_name",
               "agency_name", 
               "population", "density", "sq_miles", # is this uza's info?
               #ntd agency name is not source_agency that is presented in ridership reports
               # that agency name is the name that comes in the report, but differs from the dim_table's agency_name, confusing!
               "reporter_type", "reporting_module",]
ntd_url = f"{PROCESSED_GCS}test_bridge_with_ntd.parquet"
ntd_profile_df = pd.read_parquet(
    ntd_url, 
    filesystem = gcsfs.GCSFileSystem(),
    #filters=[[("analysis_name", "==", analysis_name)]],
    columns = ntd_columns
).drop_duplicates().reset_index(drop=True)

In [ ]:
operator_df.dtypes

In [ ]:
m1 = pd.merge(
    operator_df, 
    ntd_profile_df.rename(columns = {"analysis_name": "Analysis Name"}), 
    on = "Analysis Name", 
    how = "left",
    indicator=True
)

m1._merge.value_counts()

In [ ]:
try:
    service_area = formatted(int(ntd_profile_df.service_area_sq_miles.values[0]))
    service_pop = formatted(int(ntd_profile_df.service_area_pop.values[0]))
except:
    pass

In [ ]:
try:
    display(
        Markdown(
            f"""{analysis_name} is headquartered in <b>{ntd_profile_df.hq_county.values[0]}</b> 
            County in the Urbanized Area of <b>{ntd_profile_df.primary_uza_name.values[0]}</b>.<br>
            This operator provides <b>{service_area}</b> square miles of public transit service, which has a 
            service population of <b>{service_pop}</b>.<br>
            This organization is a {ntd_profile_df.reporter_type.values[0]}.<br>
            <b>Data Source</b>: <a href="https://www.transit.dot.gov/ntd/data-product/2022-annual-database-agency-information">National Transit Database</a> Annual Agency Information.
            """
        )
    )
except:
    pass

## NTD agency profile 
* current use case: for each `analysis_name`, label the relevant NTD agency information
   * for LA Metro, since `analysis_name` is 1 agency, we should be able to link to the single NTD ID
   * should be 1:1 matching based on `bridge_gtfs_analysis_name_x_ntd`
   * the need to dedupe dimension tables here might lead to inconsistent labeling with more use cases added.
   * already with 2 tables, 3 steps to use, there are 3 levels of deduping, before we even see the label used. likely, if we switch up order of operations, we won't get the same results.
* future use case: combine this with NTD agency grain tables (`mart_ntd_annual_reporting` or NTD Transit Supply and Demand work Shweta is working on)

**download 2 NTD tables, merge together**

1. `load_ntd`
`mart_ntd.dim_annual_agency_information`

* dedupe (with sorting) by `agency_name`
* still need to dedupe, this one `_is_current` flag is not on, it is dim table. 

2. `load_mobility`

`mart_ntd.dim_mobility_mart_providers`
* dedupe (with sorting) by `agency_name`
* still need to dedupe, even though `_is_current` flag is on within parent tables of `dim_mobility_mart_providers`.
* dimension tables get handled differently throughout analysis.

3. merge the 2 tables together.

* merge on `agency_name` still results in needing to dedupe. why?

**NTD columns used in GTFS Digest, grouped by dbt parent**

4. `dim_annual_agency_information`
* `agency_name`: used to dedupe
* `service_area_sq_miles`, `service_area_pop`, `primary_uza_name`

5. `dim_mobility_mart_providers`
* `agency_name`: used to dedupe and also join with `dim_annual_agency_information`
* `hq_county`: this column is actually just `county_geography_name`, which is already present in `bridge_gtfs_analysis_name_x_ntd`

    ```
    orgs_x_hq AS (
       SELECT * FROM {{ ref('bridge_organizations_x_headquarters_county_geography') }}
       WHERE _is_current
    
    `orgs_x_hq.county_geography_name AS hq_county`
    ```